In [1]:
from google.cloud import bigquery
import plotly.graph_objects as go
import pandas as pd

In [2]:
# 1. Fetch your data directly from BigQuery
client = bigquery.Client(project="gannett-datascience")
query = """
select
  cohort,
  contact_channel as contact_channel_1st,
  case 
    when contact_channel='Called-In first' then 'Then Online Cancel Flow'
    when contact_channel='Online first' then 'Then Called-In Cancel Flow'
    else contact_channel
  end as contact_channel_2nd,
  billing_account
from `gannett-datascience.test_results_zone.ss_test_result_v3-1`
"""
df = client.query(query).to_dataframe()

/Users/YKang1/PycharmProjects/stop_save_pChurn/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/YKang1/PycharmProjects/stop_save_pChurn/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [3]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35701 entries, 0 to 35700
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   cohort               35701 non-null  object
 1   contact_channel_1st  35701 non-null  object
 2   contact_channel_2nd  35701 non-null  object
 3   billing_account      35701 non-null  object
dtypes: object(4)
memory usage: 1.1+ MB


,cohort,contact_channel_1st,contact_channel_2nd,billing_account
0,Three-Offer Cohort,No Action yet,No Action yet,g-s1668564380
1,Two-Offer Cohort,No Action yet,No Action yet,g-s9948368128
2,Three-Offer Cohort,No Action yet,No Action yet,g-s0745130463
3,Two-Offer Cohort,No Action yet,No Action yet,g-s4650331615
4,Three-Offer Cohort,No Action yet,No Action yet,g-s7025396238


In [19]:
# 2. Transform into a continuous node list for Plotly
# (This map dynamically handles as many stages as you need)
stages = ['cohort', 'contact_channel_1st', 'contact_channel_2nd']
links = []

for i in range(len(stages)-1):
    temp_df = df.groupby([stages[i], stages[i+1]])['billing_account'].nunique().reset_index()
    temp_df.columns = ['source', 'target', 'value']
    links.append(temp_df)

sankey_df = pd.concat(links, axis=0)
sankey_df = sankey_df[sankey_df['source'] != sankey_df['target']]  # Remove self-links if any
mapping = {
    'Three-Offer Cohort': 'Three-Variant Test',
    'Two-Offer Cohort': 'Two-Variant Test',
    'Called-In first': 'Called-In Cancel Flow',
    'Online first': 'Online Cancel Flow',
}
sankey_df['source'] = sankey_df['source'].map(mapping)
sankey_df['target'] = sankey_df['target'].map(mapping).fillna(sankey_df['target'])  # Keep unmapped values as is
sankey_df

,source,target,value
0,Three-Variant Test,Called-In Cancel Flow,677
1,Three-Variant Test,Called-In Cancel Flow,1
2,Three-Variant Test,No Action yet,17627
3,Three-Variant Test,Online Cancel Flow,431
4,Three-Variant Test,Online Cancel Flow,6
5,Two-Variant Test,Called-In Cancel Flow,403
6,Two-Variant Test,No Action yet,16194
7,Two-Variant Test,Online Cancel Flow,357
8,Two-Variant Test,Online Cancel Flow,4
1,Called-In Cancel Flow,Then Online Cancel Flow,1


In [20]:
# 3. Create unique node indices
all_nodes = list(set(sankey_df['source']).union(set(sankey_df['target'])))
node_dict = {node: i for i, node in enumerate(all_nodes)}
print(node_dict)

sankey_df['source_idx'] = sankey_df['source'].map(node_dict)
sankey_df['target_idx'] = sankey_df['target'].map(node_dict)
# sankey_df['value_normalized'] = sankey_df['value'] ** 0.3

sankey_df

{'No Action yet': 0, 'Two-Variant Test': 1, 'Online Cancel Flow': 2, 'Three-Variant Test': 3, 'Then Called-In Cancel Flow': 4, 'Called-In Cancel Flow': 5, 'Then Online Cancel Flow': 6}


,source,target,value,source_idx,target_idx
0,Three-Variant Test,Called-In Cancel Flow,677,3,5
1,Three-Variant Test,Called-In Cancel Flow,1,3,5
2,Three-Variant Test,No Action yet,17627,3,0
3,Three-Variant Test,Online Cancel Flow,431,3,2
4,Three-Variant Test,Online Cancel Flow,6,3,2
5,Two-Variant Test,Called-In Cancel Flow,403,1,5
6,Two-Variant Test,No Action yet,16194,1,0
7,Two-Variant Test,Online Cancel Flow,357,1,2
8,Two-Variant Test,Online Cancel Flow,4,1,2
1,Called-In Cancel Flow,Then Online Cancel Flow,1,5,6


In [21]:
# 4. Generate the perfect interactive Sankey Chart
# Calculate the true, un-normalized total users for each node
# node_totals = []
# for node in all_nodes:
#     outgoing_flow = sankey_df[sankey_df['source'] == node]['value'].sum()
#     incoming_flow = sankey_df[sankey_df['target'] == node]['value'].sum()
#     node_totals.append(int(max(outgoing_flow, incoming_flow)))

fig = go.Figure(
    data=[
        go.Sankey(
            node = dict(
                pad = 15, 
                thickness = 20, 
                line = dict(color = "black", width = 0.5), 
                label = all_nodes,
                hovertemplate='%{label} has total %{value} users<extra></extra>',
                align = "left",
            ),
            link = dict(
                arrowlen = 50,
                source = sankey_df['source_idx'], 
                target = sankey_df['target_idx'], 
                value = sankey_df['value'],
                # customdata = sankey_df['value'],
                hovertemplate='From %{source.label} to %{target.label}:<br /> %{value} users<extra></extra>',
            )
        )
    ]
)

fig.show()

fig.update_layout(
    title_text="Subscriber Cancellation Journey Flow (Normalized Scale for Minority Paths)",
    font_size=12,
    width=900,
    height=700,
)

fig.write_html("sankey_interactive.html", include_plotlyjs="cdn", full_html=True)
print("Sankey chart exported as 'sankey_interactive.html'")

Sankey chart exported as 'sankey_interactive.html'


,source,target,value
0,"Three-Variant Test:\n18,742",Called-In Cancel Flow,677
1,"Three-Variant Test:\n18,742",Called-In first,1
2,"Three-Variant Test:\n18,742",No Action yet,17627
3,"Three-Variant Test:\n18,742",Online Cancel Flow,431
4,"Three-Variant Test:\n18,742",Online first,6
5,"Two-Variant Test:\n16,958",Called-In Cancel Flow,403
6,"Two-Variant Test:\n16,958",No Action yet,16194
7,"Two-Variant Test:\n16,958",Online Cancel Flow,357
8,"Two-Variant Test:\n16,958",Online first,4
1,Called-In Cancel Flow,Then Online Cancel Flow,1
